In [ ]:
!pip install optuna-integration[tfkeras]
!pip install --upgrade wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 12.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.6 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
pydrive2 1.21.3 requires cryptogra

In [ ]:
# --- Libraries

import os
import logging

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import random
import pandas as pd
import numpy as np
import tensorflow as tf

tf.get_logger().setLevel(logging.ERROR)

import warnings

warnings.filterwarnings('ignore')

import gc
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from tensorflow.keras import mixed_precision
import optuna

import wandb
from kaggle_secrets import UserSecretsClient
from optuna.integration.wandb import WeightsAndBiasesCallback

E0000 00:00:1774057314.931630      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774057314.996249      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [ ]:
# secret Kaggle
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

In [ ]:
# Login
wandb.login(key=wandb_key)

In [ ]:
# --- Configuration

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

INPUT_ROOT = '/kaggle/input/csiro-biomass'
TRAIN_CSV  = os.path.join(INPUT_ROOT, 'train.csv')
TEST_CSV   = os.path.join(INPUT_ROOT, 'test.csv')

IMG_SIZE   = 256

BATCH_SIZE_PER_REPLICA = 16

# 2 Gpus
STRATEGY = tf.distribute.MirroredStrategy()
GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_REPLICA * STRATEGY.num_replicas_in_sync

print(STRATEGY.num_replicas_in_sync)
print(GLOBAL_BATCH_SIZE)

# Mixed precision
mixed_precision.set_global_policy('mixed_float16')
print(mixed_precision.global_policy())

I0000 00:00:1774057340.138472      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774057340.140953      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


2
32
<DTypePolicy "mixed_float16">


In [ ]:
# --- Data prep
raw_train_df = pd.read_csv(TRAIN_CSV)

# Pivot long to wide
train_df = raw_train_df.pivot(index='image_path', columns='target_name', values='target').reset_index()

# Paths (add the complete position of each image)
train_df['full_path'] = train_df['image_path'].apply(lambda x: os.path.join(INPUT_ROOT, x))

In [ ]:
raw_train_df

,sample_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,target_name,target
0,ID1011485656__Dry_Clover_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Clover_g,0.0000
1,ID1011485656__Dry_Dead_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Dead_g,31.9984
2,ID1011485656__Dry_Green_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Green_g,16.2751
3,ID1011485656__Dry_Total_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,Dry_Total_g,48.2735
4,ID1011485656__GDM_g,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,GDM_g,16.2750
...,...,...,...,...,...,...,...,...,...
1780,ID983582017__Dry_Clover_g,train/ID983582017.jpg,2015/9/1,WA,Ryegrass,0.64,9.0000,Dry_Clover_g,0.0000
1781,ID983582017__Dry_Dead_g,train/ID983582017.jpg,2015/9/1,WA,Ryegrass,0.64,9.0000,Dry_Dead_g,0.0000
1782,ID983582017__Dry_Green_g,train/ID983582017.jpg,2015/9/1,WA,Ryegrass,0.64,9.0000,Dry_Green_g,40.9400
1783,ID983582017__Dry_Total_g,train/ID983582017.jpg,2015/9/1,WA,Ryegrass,0.64,9.0000,Dry_Total_g,40.9400


In [ ]:
train_df

target_name,image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g,full_path
0,train/ID1011485656.jpg,0.0000,31.9984,16.2751,48.2735,16.2750,/kaggle/input/csiro-biomass/train/ID1011485656...
1,train/ID1012260530.jpg,0.0000,0.0000,7.6000,7.6000,7.6000,/kaggle/input/csiro-biomass/train/ID1012260530...
2,train/ID1025234388.jpg,6.0500,0.0000,0.0000,6.0500,6.0500,/kaggle/input/csiro-biomass/train/ID1025234388...
3,train/ID1028611175.jpg,0.0000,30.9703,24.2376,55.2079,24.2376,/kaggle/input/csiro-biomass/train/ID1028611175...
4,train/ID1035947949.jpg,0.4343,23.2239,10.5261,34.1844,10.9605,/kaggle/input/csiro-biomass/train/ID1035947949...
...,...,...,...,...,...,...,...
352,train/ID975115267.jpg,40.0300,0.0000,0.8000,40.8300,40.8300,/kaggle/input/csiro-biomass/train/ID975115267.jpg
353,train/ID978026131.jpg,24.6445,4.1948,12.0601,40.8994,36.7046,/kaggle/input/csiro-biomass/train/ID978026131.jpg
354,train/ID980538882.jpg,0.0000,1.1457,91.6543,92.8000,91.6543,/kaggle/input/csiro-biomass/train/ID980538882.jpg
355,train/ID980878870.jpg,32.3575,0.0000,2.0325,34.3900,34.3900,/kaggle/input/csiro-biomass/train/ID980878870.jpg


In [ ]:
# Scale Targets

target_cols = ['Dry_Total_g', 'GDM_g', 'Dry_Green_g']   # Only these 3 bcs they should be the easist to predic (the others with math)
MAX_VAL = train_df[target_cols].max().max()             # Max value for scaling
TARGET_SCALER = MAX_VAL * 1.1

# Arrays
X_paths = train_df['full_path'].values
Y_targets = train_df[target_cols].values.astype('float32') / TARGET_SCALER   # Scaling

In [ ]:
print(X_paths[:5])
print("\n")
print(Y_targets[:5])

['/kaggle/input/csiro-biomass/train/ID1011485656.jpg'
 '/kaggle/input/csiro-biomass/train/ID1012260530.jpg'
 '/kaggle/input/csiro-biomass/train/ID1025234388.jpg'
 '/kaggle/input/csiro-biomass/train/ID1028611175.jpg'
 '/kaggle/input/csiro-biomass/train/ID1035947949.jpg']


[[0.23632202 0.07967396 0.07967445]
 [0.03720566 0.03720566 0.03720566]
 [0.02961766 0.02961766 0.        ]
 [0.27026924 0.11865472 0.11865472]
 [0.16734909 0.05365692 0.05153033]]


In [ ]:
X_tr, X_v, Y_tr, Y_v = train_test_split(
    X_paths,
    Y_targets,
    test_size=0.2,
    random_state=42 # reproducible
)

print(f"Training samples: {len(X_tr)}")
print(f"Validation samples: {len(X_v)}")

Training samples: 285
Validation samples: 72


In [ ]:
# --- Load & pre-process

@tf.function
def load_image(path, label):
    # Load
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels = 3)
    # Resize
    img = tf.image.resize(img,
                          [IMG_SIZE, IMG_SIZE],
                          preserve_aspect_ratio = False,
                          antialias = True
                         )
    # cast
    img = tf.cast(img, tf.float32)
    return img, label





def create_augmenter(zoom, brightness=0.0, contrast=0.0):
    zoom_layer = keras.layers.RandomZoom(
        height_factor=(0.0, zoom),
        width_factor=(0.0, zoom),
        fill_mode='constant'
    )

    @tf.function
    def augment(img, label):
        # Flips
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)

        # Rotations
        k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
        img = tf.image.rot90(img, k)

        # Zoom
        img = tf.expand_dims(img, 0)
        img = zoom_layer(img)
        img = tf.squeeze(img, 0)

        # Brightness and contrast (not good to add)
        if brightness > 0:
            img = tf.image.random_brightness(img, brightness)
        if contrast > 0:
            img = tf.image.random_contrast(img, 1-contrast, 1+contrast)

        return img, label

    return augment


def augment(img, label):
    # Flips
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)

    # Rotations
    k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
    img = tf.image.rot90(img, k)

    return img, label

In [ ]:
# --- MixUp & Dataset Prep

def create_mixup_mapper(alpha):
    """ MixUp layer"""
    mixup_layer = keras.layers.MixUp(alpha=alpha, dtype='float32')

    @tf.function
    def apply_mixup(images, labels):
        images = tf.cast(images, tf.float32)
        labels = tf.cast(labels, tf.float32)
        inputs = {"images": images, "labels": labels}
        outputs = mixup_layer(inputs)
        return outputs["images"], outputs["labels"]

    return apply_mixup


def prepare_dataset(paths, targets, batch_size, augment_fn=None, mixup_alpha=0.3):
    ds = tf.data.Dataset.from_tensor_slices((paths, targets))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

    if augment_fn is not None:
        ds = ds.shuffle(2048)
        ds = ds.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    if augment_fn is not None and mixup_alpha > 0.0:
        ds = ds.map(create_mixup_mapper(mixup_alpha), num_parallel_calls=tf.data.AUTOTUNE)

    return ds

In [ ]:
# Generalized mean pooling (GeM)

class GeMPooling2D(layers.Layer):
    def __init__(self, p_init=3.0, **kwargs):
        super().__init__(**kwargs)
        self.p_init = p_init

    def build(self, input_shape):
        self.p = self.add_weight(
            name="p_param",
            shape=(1,),
            initializer=keras.initializers.Constant(self.p_init),
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        x = tf.maximum(inputs, 1e-6)
        x = tf.pow(x, self.p)
        x = tf.reduce_mean(x, axis=[1, 2], keepdims=False)
        return tf.pow(x, 1.0 / self.p)

In [ ]:
def attention_block(inputs, ratio=8):
    """
    Squeeze-and-Excitation Block
    """
    # Number of features
    channels = inputs.shape[-1]
    # Squeeze: global Average Pooling to get a summary vector (Squeeze feature from (H, W, C) to (1, 1, C))
    x = layers.GlobalAveragePooling2D(keepdims=True)(inputs)
    # Excitation: bottleneck part, reduce param to learn (weights)
    x = layers.Dense(channels // ratio, activation='swish')(x)
    # Restoration: restore to the original dimension and weights can be put into the original feature map
    x = layers.Dense(channels, activation='sigmoid')(x)
    # Scale: multiply the original inputs by the calculated weights
    return layers.Multiply()([inputs, x])

In [ ]:
# --- Optuna Objective

weights_path = '/kaggle/input/models/marcorosato/convnext-small/tensorflow2/default/1/convnext_small_notop.h5'

def objective(trial):
    keras.backend.clear_session()

    tf.random.set_seed(42)
    np.random.seed(42)

    gc.collect()

    # Architecture
    #head_type = trial.suggest_categorical("head_type", ["single", "multi"])
    dense_units = trial.suggest_categorical("shared_dense_units", [64, 128, 256, 384])
    dense_units_PREHEADS = trial.suggest_categorical("prehead_dense_units", [64, 128, 256])
    dense_units_HEADS = trial.suggest_categorical("head_dense_units", [64, 128])
    #activation = trial.suggest_categorical("activation", ["gelu", "swish", "relu"])
    #pooling_type = trial.suggest_categorical("pooling_type", ["avg", "max", "concat"])
    #unfreeze_level = trial.suggest_int("unfreeze_level", 0, 2)

    # Attention Block & GeM
    #use_attention = trial.suggest_categorical("use_attention", [True, False])
    attention_ratio = trial.suggest_categorical("attention_ratio", [4, 8, 16])

    gem_p_init = trial.suggest_float("gem_p_init", 1.0, 4.0)


    # Regularization & Augmentation
    dropout_rate = trial.suggest_float("shared_dropout", 0.1, 0.5)
    dropout_rate_HEADS = trial.suggest_float("head_dropout", 0.1, 0.5)
    noise_level = trial.suggest_float("noise_level", 0.0, 0.2)
    mixup_alpha = trial.suggest_float("mixup_alpha", 0.0, 0.4)
    #use_zoom = trial.suggest_categorical("use_zoom", [True, False])

    #use_brightness = trial.suggest_categorical("use_brightness", [True, False])
    #brightness = trial.suggest_float("brightness", 0.0, 0.2) if use_brightness else 0.0
    #use_contrast = trial.suggest_categorical("use_contrast", [True, False])
    #contrast = trial.suggest_float("contrast", 0.0, 0.2) if use_contrast else 0.0

    # Optimizer & Loss
    lr = trial.suggest_float("lr", 5e-5, 1e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    decay_steps = trial.suggest_int("decay_steps", 1000, 5000)
    huber_delta = trial.suggest_float("huber_delta", 0.1, 3.0)

    # Datasets
    current_batch_size = GLOBAL_BATCH_SIZE

    #zoom_val = trial.suggest_float("zoom", 0.0, 0.25) if use_zoom else 0.0
    #augment_fn = create_augmenter(zoom_val, brightness, contrast)

    train_ds = prepare_dataset(X_tr, Y_tr, current_batch_size, augment_fn=augment, mixup_alpha=mixup_alpha)
    val_ds   = prepare_dataset(X_v,  Y_v,  current_batch_size)

    # Build Model
    with STRATEGY.scope():
        inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

        base = keras.applications.ConvNeXtSmall(
            include_top=False,
            weights=None,
            input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )
        base.load_weights(weights_path)

        # Fine-tuning Logic
        base.trainable = True
        #if unfreeze_level == 0:
        #    base.trainable = False
        #elif unfreeze_level == 1:
        #    for layer in base.layers[:-20]:
        #        layer.trainable = False

        x = base(inputs)

        # Optional Attention
        #if use_attention:
        #    x = attention_block(x, ratio=attention_ratio)

        x = attention_block(x, ratio=attention_ratio)

        # Pooling
        #if pooling_type == "avg":
       #    x = layers.GlobalAveragePooling2D()(x)
       #elif pooling_type == "max":
       #    x = layers.GlobalMaxPooling2D()(x)
       #else:
       #    x_avg = layers.GlobalAveragePooling2D()(x)
       #    x_max = layers.GlobalMaxPooling2D()(x)
        #    x = layers.Concatenate()([x_avg, x_max])

        x = GeMPooling2D(p_init=gem_p_init)(x)

        # Regularization
        if noise_level > 0:
            x = layers.GaussianNoise(noise_level)(x)

        x = layers.Dense(dense_units, activation='swish')(x)
        x = layers.Dropout(dropout_rate)(x)

        # --- Heads ---
        shared = layers.Dense(dense_units_PREHEADS, activation='swish')(x)

        #if head_type == "single":
            # One path for all 3
            #h_single = layers.Dense(64, activation='swish')(shared)
            #outputs = layers.Dense(3, activation='linear', dtype='float32')(h_single)
        #else:
            # Head 1
            #h1_total = layers.Dense(64, activation='swish')(shared)
            #out1 = layers.Dense(1, activation='linear', name='total_head')(h1_total)

            # Head 2
            #h1_gdm = layers.Dense(64, activation='swish')(shared)
            #out2 = layers.Dense(1, activation='linear', name='gdm_head')(h1_gdm)

            # Head 3
            #h1_green = layers.Dense(64, activation='swish')(shared)
            #out3 = layers.Dense(1, activation='linear', name='green_head')(h1_green)

            #outputs = layers.Concatenate(axis=-1, dtype='float32')([out1, out2, out3])


        # Head 1
        h1_total = layers.Dense(dense_units_HEADS, activation='swish')(shared)
        drop_h1 = layers.Dropout(dropout_rate_HEADS)(h1_total)
        out1 = layers.Dense(1, activation='linear', name='total_head')(drop_h1)

        # Head 2
        h1_gdm = layers.Dense(dense_units_HEADS, activation='swish')(shared)
        drop_h2 = layers.Dropout(dropout_rate_HEADS)(h1_gdm)
        out2 = layers.Dense(1, activation='linear', name='gdm_head')(drop_h2)

        # Head 3
        h1_green = layers.Dense(dense_units_HEADS, activation='swish')(shared)
        drop_h3 = layers.Dropout(dropout_rate_HEADS)(h1_green)
        out3 = layers.Dense(1, activation='linear', name='green_head')(drop_h3)

        outputs = layers.Concatenate(axis=-1, dtype='float32')([out1, out2, out3])

        model = keras.Model(inputs, outputs)

        # Compilation
        steps_per_epoch = len(X_tr) // current_batch_size
        warmup_steps = steps_per_epoch * 2

        lr_schedule = keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=lr,
            decay_steps=decay_steps,
            alpha=0.01,
            warmup_target=lr,
            warmup_steps=warmup_steps
        )

        model.compile(
            optimizer = keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=weight_decay),
            loss = keras.losses.Huber(delta=huber_delta),
            metrics=['mae']
        )

    # Training
    pruning_callback = optuna.integration.TFKerasPruningCallback(trial, "val_loss")
    early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=15,
        callbacks=[pruning_callback, early_stop],
        verbose=0
    )

    return min(history.history['val_loss'])

In [ ]:
import shutil

# Paths
input_db_path = "/kaggle/input/datasets/marcorosato/optuna-small/optuna_biomass_small.db"
working_db_path = "/kaggle/working/optuna_biomass_small.db"



# Writable directory
try:
    shutil.copy(input_db_path, working_db_path)
    print("Database copied successfully to /kaggle/working/")
except Exception as e:
    print(f"Error copying database: {e}")

# Load from writable
study = optuna.load_study(
    study_name=f"biomass_optimization_base",
    storage=f"sqlite:///{working_db_path}"      # "sqlite:///optuna_biomass.db"   f"sqlite:///{working_db_path}"
)

In [ ]:
# Initialize the W&B Callback
# Project "csiro-biomass" on W&B
wandbc = WeightsAndBiasesCallback(
    metric_name="value",
    wandb_kwargs={"project": "csiro-biomass-optuna-SMALL"}
)

# Create the study as usual
study = optuna.create_study(
    study_name="biomass_optimization_SMALL",
    storage="sqlite:///optuna_biomass.db",  #   f"sqlite:///{working_db_path}"    "sqlite:///optuna_biomass.db"
    load_if_exists=True,
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5)
)

print("Starting Optuna")

# Callback
study.optimize(
    objective,
    n_trials=50,
    callbacks=[wandbc] # logs cloud
)

In [ ]:
print("\nBEST HYPERPARAMETERS")
print("BEST:", study.best_params, "value:", study.best_value)
trials_df = study.trials_dataframe()
trials_df.to_csv("optuna_trials.csv", index=False)